# 0-order optimization

In [ ]:
# pip install -e .
# python -m cell_simulator.search_example

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from cell_simulator.dynamics import HarmonicOscillatorContraction
from cell_simulator.forces import ForceModel
from cell_simulator.geometry import Cell, PillarGrid
from cell_simulator.integrators import RK4
from cell_simulator.observations import TrackingObservationModel
from cell_simulator.simulator import Experiment, ExperimentConfig
from cell_simulator.visualize import *

### Setup physics

In [ ]:
set_style()
grid = PillarGrid.square_lattice(15, 15, 4000.0)
cell = Cell(center_nm=np.array([28900.0, 27400.0]), radius_nm=20000.0)
force_model = ForceModel(F_max=600.0, k=4.8)
times = np.arange(0.0, 30.0 + 1e-9, 0.5)

### Setup simulation + loss

In [ ]:
def simulate(omega, phase, observation_model=None):
    cfg = ExperimentConfig(
        pillar_grid=grid,
        cell=cell,
        dynamics=HarmonicOscillatorContraction(omega, phase),
        integrator=RK4(),
        force_model=force_model,
        observation_model=observation_model,
    )
    return Experiment(cfg).run(times, steps_per_frame=10)


def loss(result):
    return float(np.mean((result.forces - target) ** 2))

### Define GT parameters

In [ ]:
true_omega, true_phase = 2 * np.pi / 10.0, 0.8
true_data = simulate(true_omega, true_phase, TrackingObservationModel(2.0, rng=np.random.default_rng(1)))
target = true_data.observed_forces

noise_floor = loss(true_data)  # true parameters, noise-free prediction vs noisy target
print(f"Noise floor: {noise_floor:.2f} i.e. minimum achievable loss with perfect parameters")

## Simulate likelihood

In [ ]:
N = 10_000
rng = np.random.default_rng(0)

# Sample parameters from your prior/proposal distributions
omegas = np.abs(rng.normal(0.6, 0.15, N))
phis = rng.normal(0.0, 1.5, N)

# Simulate
candidates = [simulate(w, p) for w, p in zip(omegas, phis)]

# Calculate losses
losses = np.array([loss(r) for r in candidates])

# Convert loss -> likelihood
sigma = 0.1  # <-- should reflect the expected observational/simulation noise

log_likelihoods = -losses / (2 * sigma**2)

# Normalize safely
log_likelihoods -= np.max(log_likelihoods)
weights = np.exp(log_likelihoods)
weights /= np.sum(weights)

In [ ]:
omega_estimate = np.sum(weights * omegas)
phi_estimate = np.sum(weights * phis)

print("Estimated omega:", omega_estimate)
print("Estimated phi:", phi_estimate)

In [ ]:
omega_std = np.sqrt(np.sum(weights * (omegas - omega_estimate) ** 2))

phi_std = np.sqrt(np.sum(weights * (phis - phi_estimate) ** 2))

print("Omega:", omega_estimate, "+/-", omega_std)
print("Phi:", phi_estimate, "+/-", phi_std)

In [ ]:
fig = plot_parameter_search(omegas, phis, losses, true_omega, true_phase, noise_floor)

## Simulate MCMC (Metropolis–Hastings)

Target distribution: the posterior $p(\omega, \phi \mid y) \propto p(y \mid \omega, \phi)\, p(\omega)\, p(\phi)$.

**Likelihood.** The observation model adds independent Gaussian noise with standard deviation $\sigma$ to each of the $n$ observed force components, so

$$
\log p(y \mid \omega, \phi) = -\frac{1}{2\sigma^2}\sum_{j=1}^{n}\big(y_j - F_j(\omega,\phi)\big)^2 + \text{const}
= -\frac{n \cdot \mathrm{MSE}(\omega,\phi)}{2\sigma^2} + \text{const}.
$$

Two things follow: $\sigma$ is fixed by the observation model (not a free tuning knob), and because `loss` is a *mean*, it has to be multiplied by $n$.
Here $\sigma = 2.0\ \text{nm (tracking noise)} \times k = 4.8 \approx 9.6$, which should match $\sqrt{\text{noise floor}}$.

**Prior.** The same priors as in the sampling section above: $\omega \sim |\mathcal{N}(0.6, 0.15^2)|$, $\phi \sim \mathcal{N}(0, 1.5^2)$.

In [ ]:
# --- Likelihood --------------------------------------------------------------
# Noise std per force component = tracking noise (nm) x pillar stiffness.
# Must match TrackingObservationModel(2.0, ...) and ForceModel(k=4.8) above.
sigma_obs = 2.0 * 4.8
n_obs = target.size  # number of observed force components

print(f"sigma_obs = {sigma_obs:.2f},  sqrt(noise floor) = {np.sqrt(noise_floor):.2f}  (should roughly agree)")
print(f"n_obs = {n_obs}")


def log_likelihood(result):
    # -SSE / (2 sigma^2) = -n * MSE / (2 sigma^2), constants dropped
    return -0.5 * n_obs * loss(result) / sigma_obs**2


# --- Prior (same as the prior sampling section) --------------------------------
def log_prior(omega, phi):
    if omega < 0:
        return -np.inf
    # folded normal |N(0.6, 0.15)|: density ∝ N(omega; 0.6) + N(-omega; 0.6)
    lp_omega = np.logaddexp(-0.5 * ((omega - 0.6) / 0.15) ** 2, -0.5 * ((omega + 0.6) / 0.15) ** 2)
    lp_phi = -0.5 * (phi / 1.5) ** 2
    return lp_omega + lp_phi

**Sampler.** Random-walk Metropolis with Gaussian proposals. The posterior is much narrower than the prior (lots of data), so good step sizes are not known in advance.
The step sizes are therefore adapted during burn-in (scaled up when acceptance is above ~0.3, down when below) and **frozen afterwards**, so the samples after burn-in come from a standard, valid Metropolis–Hastings chain.

### Scalar fit: contraction signal $a(t)$

As a first step we fit the scalar contraction signal $a(t) = \tfrac{1}{2}\big(1 + \sin(\omega t + \varphi)\big)$ directly, before fitting the force field.
$a(t)$ is not observable in real TFM data, so here we observe it with Gaussian noise $\sigma_a = \sigma_{\text{obs}} / F_{\max} = (2.0 \cdot 4.8)/600 = 0.016$, which gives the same relative noise level as the force observations.

The likelihood is the same as above with $\sigma_a$ instead of $\sigma$ and $n_a$ = number of time points; the prior and sampler are unchanged.

In [ ]:
def contraction(omega, phase):
    # Contraction signal a(t) on the same time grid as the simulator (matches panel (a) of plot_fit_comparison)
    return 0.5 * (1 + np.sin(omega * times + phase))


sigma_a = 2.0 * 4.8 / 600.0  # tracking noise x stiffness / F_max -> same relative noise as the forces
a_true = contraction(true_omega, true_phase)
target_a = a_true + np.random.default_rng(2).normal(0.0, sigma_a, a_true.shape)


def loss_a(omega, phase):
    return float(np.mean((contraction(omega, phase) - target_a) ** 2))


noise_floor_a = loss_a(true_omega, true_phase)
print(f"Noise floor (a): {noise_floor_a:.2e}  (sigma_a^2 = {sigma_a**2:.2e}, should roughly agree)")

In [ ]:
n_a = target_a.size


def log_post_a(omega, phi):
    lp = log_prior(omega, phi)
    if not np.isfinite(lp):
        return -np.inf
    return lp - 0.5 * n_a * loss_a(omega, phi) / sigma_a**2


def run_mh(
    log_post,
    omega0,
    phi0,
    N=10_000,
    burn=2_000,
    omega_step=0.001,
    phi_step=0.01,
    target_acceptance=0.3,
    adapt_every=100,
    seed=0,
):
    # Random-walk Metropolis-Hastings, step sizes adapted during burn-in only (same scheme as the force-field chain)
    rng = np.random.default_rng(seed)
    omegas, phis = np.empty(N), np.empty(N)
    omega_c, phi_c = omega0, phi0
    logp_c = log_post(omega_c, phi_c)
    accepted = accepted_window = 0
    for i in range(N):
        omega_p = omega_c + rng.normal(0, omega_step)
        phi_p = phi_c + rng.normal(0, phi_step)
        logp_p = log_post(omega_p, phi_p)
        if np.log(rng.random()) < logp_p - logp_c:
            omega_c, phi_c, logp_c = omega_p, phi_p, logp_p
            accepted_window += 1
            accepted += i >= burn
        omegas[i], phis[i] = omega_c, phi_c
        if i < burn and (i + 1) % adapt_every == 0:
            scale = np.exp(2 * (accepted_window / adapt_every - target_acceptance))
            omega_step *= scale
            phi_step *= scale
            accepted_window = 0
    print(f"Final step sizes: omega {omega_step:.2e}, phi {phi_step:.2e}")
    print("Acceptance rate after burn-in:", accepted / (N - burn))
    return omegas, phis


burn_a = 2_000
omegas_a, phis_a = run_mh(log_post_a, 0.6, 0.0, burn=burn_a)
omega_samples_a, phi_samples_a = omegas_a[burn_a:], phis_a[burn_a:]

print("\nOmega estimate:", omega_samples_a.mean(), "  (true:", true_omega, ")  SD:", omega_samples_a.std())
print("Phi estimate:", phi_samples_a.mean(), "  (true:", true_phase, ")  SD:", phi_samples_a.std())
print("Omega 95% interval:", np.percentile(omega_samples_a, [2.5, 97.5]))
print("Phi 95% interval:", np.percentile(phi_samples_a, [2.5, 97.5]))
print("Correlation(omega, phi):", np.corrcoef(omega_samples_a, phi_samples_a)[0, 1])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), gridspec_kw={"width_ratios": [3, 1]})
it = np.arange(burn_a, len(omegas_a))
for row, (s, true, name) in enumerate([(omega_samples_a, true_omega, "Omega"), (phi_samples_a, true_phase, "Phi")]):
    axes[row, 0].plot(it, s, lw=0.6)
    axes[row, 0].axhline(true, color="k", lw=0.8, label="True")
    axes[row, 0].set_ylabel(name)
    x = s - s.mean()
    acf = np.correlate(x, x, mode="full")[len(x) - 1 :][:200] / (x @ x)
    axes[row, 1].plot(acf)
    axes[row, 1].axhline(0, color="k", lw=0.5)
    axes[row, 1].set_title(f"ACF {name}")
axes[0, 0].legend()
axes[1, 0].set_xlabel("MCMC iteration")
fig.suptitle("Scalar fit on a(t), after burn-in")
plt.tight_layout()
plt.show()

### Force-field fit

In [ ]:
# Keep the prior samples, the MCMC cells below overwrite omegas/phis/losses
prior_omegas, prior_phis, prior_losses = omegas.copy(), phis.copy(), losses.copy()

In [ ]:
rng = np.random.default_rng(0)

N = 10_000  # total iterations
burn = 2_000  # burn-in (also the adaptation phase)

# Starting parameters
omega_current = 0.6
phi_current = 0.0

# Initial proposal step sizes (adapted during burn-in)
omega_step = 0.001
phi_step = 0.01
target_acceptance = 0.3
adapt_every = 100

# Storage
omegas = np.empty(N)
phis = np.empty(N)
losses = np.empty(N)
log_posts = np.empty(N)

# Starting point
result_current = simulate(omega_current, phi_current)
loss_current = loss(result_current)
logp_current = log_likelihood(result_current) + log_prior(omega_current, phi_current)

accepted = 0  # accepted proposals after burn-in
accepted_window = 0  # accepted proposals in the current adaptation window

for i in range(N):
    # Propose a nearby parameter combination (symmetric proposal -> MH ratio is just the posterior ratio)
    omega_proposed = omega_current + rng.normal(0, omega_step)
    phi_proposed = phi_current + rng.normal(0, phi_step)

    lp_prior = log_prior(omega_proposed, phi_proposed)

    # Outside the support (omega < 0) the posterior is 0 -> reject without simulating
    if np.isfinite(lp_prior):
        result_proposed = simulate(omega_proposed, phi_proposed)
        loss_proposed = loss(result_proposed)
        logp_proposed = log_likelihood(result_proposed) + lp_prior

        # Metropolis-Hastings acceptance
        if np.log(rng.random()) < logp_proposed - logp_current:
            omega_current, phi_current = omega_proposed, phi_proposed
            loss_current, logp_current = loss_proposed, logp_proposed
            accepted_window += 1
            if i >= burn:
                accepted += 1

    # Store current state (a rejection repeats the current state -- that is part of MH)
    omegas[i] = omega_current
    phis[i] = phi_current
    losses[i] = loss_current
    log_posts[i] = logp_current

    # Adapt step sizes during burn-in only
    if i < burn and (i + 1) % adapt_every == 0:
        rate = accepted_window / adapt_every
        scale = np.exp(2 * (rate - target_acceptance))
        omega_step *= scale
        phi_step *= scale
        accepted_window = 0

print(f"Final step sizes: omega {omega_step:.2e}, phi {phi_step:.2e}")
print("Acceptance rate after burn-in:", accepted / (N - burn))

In [ ]:
omega_samples = omegas[burn:]
phi_samples = phis[burn:]
loss_samples = losses[burn:]

print("Loss after burn-in:")
print("min:", np.min(loss_samples), " max:", np.max(loss_samples))
print("mean:", np.mean(loss_samples), " median:", np.median(loss_samples), " std:", np.std(loss_samples))
print("(noise floor:", noise_floor, ")")

print("\nParameter ranges after burn-in:")
print("omega:", np.min(omega_samples), np.max(omega_samples))
print("phi:", np.min(phi_samples), np.max(phi_samples))

In [ ]:
omega_mean = np.mean(omega_samples)
phi_mean = np.mean(phi_samples)
iterations = np.arange(N)


fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Omega
axes[0].plot(iterations, omegas, alpha=0.5)
axes[0].axvline(burn, linestyle="--", label="Burn-in")
axes[0].axhline(true_omega, color="k", linewidth=0.8, label=f"True = {true_omega:.3f}")
axes[0].axhline(omega_mean, linestyle=":", label=f"Posterior mean = {omega_mean:.4f}")
axes[0].set_ylabel("Omega")
axes[0].legend()

# Phi
axes[1].plot(iterations, phis, alpha=0.5)
axes[1].axvline(burn, linestyle="--", label="Burn-in")
axes[1].axhline(true_phase, color="k", linewidth=0.8, label=f"True = {true_phase:.3f}")
axes[1].axhline(phi_mean, linestyle=":", label=f"Posterior mean = {phi_mean:.4f}")
axes[1].set_xlabel("MCMC iteration")
axes[1].set_ylabel("Phi")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("Omega estimate:", np.mean(omega_samples), "  (true:", true_omega, ")")
print("Omega SD:", np.std(omega_samples))

print("Phi estimate:", np.mean(phi_samples), "  (true:", true_phase, ")")
print("Phi SD:", np.std(phi_samples))

omega_ci = np.percentile(omega_samples, [2.5, 97.5])
phi_ci = np.percentile(phi_samples, [2.5, 97.5])

print("Omega 95% interval:", omega_ci)
print("Phi 95% interval:", phi_ci)
print("Correlation(omega, phi):", np.corrcoef(omega_samples, phi_samples)[0, 1])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), gridspec_kw={"width_ratios": [3, 1]})
it = np.arange(burn, N)
for row, (s, true, name) in enumerate([(omega_samples, true_omega, "Omega"), (phi_samples, true_phase, "Phi")]):
    axes[row, 0].plot(it, s, lw=0.6)
    axes[row, 0].axhline(true, color="k", lw=0.8, label="True")
    axes[row, 0].set_ylabel(name)
    # autocorrelation up to lag 200
    x = s - s.mean()
    acf = np.correlate(x, x, mode="full")[len(x) - 1 :][:200] / (x @ x)
    axes[row, 1].plot(acf)
    axes[row, 1].axhline(0, color="k", lw=0.5)
    axes[row, 1].set_title(f"ACF {name}")
axes[0, 0].legend()
axes[1, 0].set_xlabel("MCMC iteration")
plt.tight_layout()
plt.show()

In [ ]:
def ess(x, max_lag=500):
    x = x - x.mean()
    acf = np.correlate(x, x, mode="full")[len(x) - 1 :][:max_lag] / (x @ x)
    cut = np.argmax(acf < 0) if np.any(acf < 0) else max_lag  # sum until the ACF first goes negative
    return len(x) / (1 + 2 * acf[1:cut].sum())


print("ESS omega:", ess(omega_samples), " ESS phi:", ess(phi_samples))

In [ ]:
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(14, 6))
gs = GridSpec(2, 2, figure=fig, width_ratios=[1, 1.4])
ax_loss = fig.add_subplot(gs[:, 0])
ax_omega = fig.add_subplot(gs[0, 1])
ax_phi = fig.add_subplot(gs[1, 1], sharex=ax_omega)

# --- Left: prior samples coloured by loss, with the MCMC path on top ---------
order = np.argsort(prior_losses)[::-1]  # draw low-loss points last so they sit on top
sc = ax_loss.scatter(
    prior_omegas[order], prior_phis[order], c=np.log10(prior_losses[order]), cmap="viridis_r", s=8, linewidths=0
)
fig.colorbar(sc, ax=ax_loss, label=r"$\log_{10}$ MSE")

ax_loss.plot(omegas[:burn], phis[:burn], color="white", lw=1.0, alpha=0.9, label="MCMC burn-in")
ax_loss.plot(omegas[burn:], phis[burn:], color="tab:red", lw=0.5, alpha=0.8, label="MCMC after burn-in")
ax_loss.plot(omegas[0], phis[0], "o", color="white", mec="k", ms=6, label="Start")
ax_loss.plot(true_omega, true_phase, "*", color="gold", mec="k", ms=12, label="Truth")
ax_loss.set_xlabel(r"$\omega$ [rad/s]")
ax_loss.set_ylabel(r"$\varphi$ [rad]")
ax_loss.legend(loc="upper right", fontsize=8)

axins = ax_loss.inset_axes([0.62, 0.05, 0.35, 0.35])
axins.plot(omegas[burn:], phis[burn:], color="tab:red", lw=0.4)
axins.plot(true_omega, true_phase, "*", color="gold", mec="k", ms=10)
axins.tick_params(labelsize=6)
ax_loss.indicate_inset_zoom(axins, edgecolor="k")

# --- Right: trace plots ------------------------------------------------------
iterations = np.arange(len(omegas))
for ax, s, true, name in [(ax_omega, omegas, true_omega, r"$\omega$"), (ax_phi, phis, true_phase, r"$\varphi$")]:
    ax.plot(iterations, s, lw=0.6, alpha=0.8)
    ax.axvline(burn, linestyle="--", color="k", lw=0.8, label="Burn-in")
    ax.axhline(true, color="gold", lw=1.2, label="Truth")
    ax.set_ylabel(name)
ax_omega.legend(loc="upper right", fontsize=8)
ax_omega.tick_params(labelbottom=False)
ax_phi.set_xlabel("MCMC iteration")

plt.tight_layout()
plt.show()